# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### The research question

**Question:** Given a page's search performance in one month, can we rank
which pages are most likely to lose search visibility the following month —
well enough to help a content editor decide which pages to review first?

**Unit of analysis:** one content page per client.

**Decision supported:** which pages a content editor reviews and
potentially refreshes first, out of a portfolio too large to review
manually.

**Action:** a content editor uses the ranked output as a starting
shortlist, not a final verdict.

**Cost of a wrong call:** wasted editor hours reviewing a page that did not
need it, or a genuinely declining page left unreviewed.

**Why ML, not a fixed rule:** a single threshold rule (Week 4 baseline)
already does reasonably — but a learned model can combine several
correlated signals (impressions, clicks, position) in a way a human-written
rule cannot easily replicate, and comparing the two honestly is the point
of this project.

In [5]:
frame = {
    "decision": "Which pages should be reviewed first for content refresh?",
    "actor": "Content editor",
    "cost_of_wrong_call": "Wasted review time OR a missed real decline",
    "why_ml": "Combines 3 correlated signals; compared against the Week-4 rule baseline"
}
for k, v in frame.items():
    print(f"{k}: {v}")

decision: Which pages should be reviewed first for content refresh?
actor: Content editor
cost_of_wrong_call: Wasted review time OR a missed real decline
why_ml: Combines 3 correlated signals; compared against the Week-4 rule baseline


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

**Release:** FlyRank/internship-warehouse (Hugging Face), table
`fact_content_daily_performance`.

**Tables and windows used:**
- `month=2026-03` (9,841,378 page-day rows) — feature window
- `month=2026-04` (10,424,730 page-day rows) — outcome window

Both are mid-panel months; the final month (`2026-06`, the `_sample`
table) was never used, per the sealed-test-month rule.

**Aggregation:** page-day rows were summed to one row per
(client, page) per month — 176,738 pages after keeping only pages with
march_impressions > 0.

**Excluded, with reasons:**
- `trend_direction`, `trend_pct` — rule-derived columns; using them as a
  label or feature risks the model re-learning a threshold rule instead of
  finding real signal (the "label trap" from Week 2/3).
- `ai_*` referral columns — mostly missing outside GA4-available rows, not
  needed for this lane's core question.
- All client and content identifiers are pseudonymized hashes, used only
  for grouping/splitting, never as features or displayed values.

No client names, raw queries, or private data appear anywhere in this
notebook or the published paper.

In [1]:
# Public-safe data summary (recomputed from Week 3/5/6 pipeline)
print("March partition rows (page-days):", 9841378)
print("April partition rows (page-days):", 10424730)
print("Pages after aggregation and filtering:", 176738)
print("Train clients:", 36, "| Test clients:", 11, "| Shared clients:", 0)

March partition rows (page-days): 9841378
April partition rows (page-days): 10424730
Pages after aggregation and filtering: 176738
Train clients: 36 | Test clients: 11 | Shared clients: 0


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

**Label:** target_decline = 1 if a page's April impressions were lower
than its March impressions, else 0. This is an observed cross-month
outcome, not a rule-derived category — it avoids the label trap identified
in Week 2/3.

**Features (all knowable before the label window):**
- march_impressions — total March search impressions
- march_clicks — total March clicks
- march_position — traffic-weighted average March ranking position

**Baseline (Week 4):** a transparent rule, score = 2 * log1p(impressions),
with a reason code (HIGH_VISIBILITY_OPPORTUNITY) for pages with high
impressions and a rankable position. No fitted weights.

**Model:** a depth-3 Decision Tree Classifier (scikit-learn), chosen for
interpretability over marginal accuracy gains from more complex methods.

**Validation design:** GroupShuffleSplit by client_hash_id (test_size=0.22,
random_state=42) — no client appears in both train and test. A naive
random split was also run for comparison (Week 6) to make the value of
grouping explicit, not just assumed.

**Leakage checks performed (Week 3 and Week 6):**
- Deliberately injected a label-derived feature and confirmed the
  evaluation harness detects it, in two independent tests on real
  warehouse data:
  - Week 3: injected gsc_clicks (the column a quick proxy label was
    directly computed from) — score jumped from 0.876 to 1.0 (ROC-AUC).
  - Week 6: injected april_impressions (which directly determines the
    real cross-month label) — score jumped from 0.611 to 0.991 (Average
    Precision).
- Confirmed zero row-level overlap between train and test after grouping.
- Checked feature-target correlations for suspiciously high values
  (all near zero: 0.008, -0.036, -0.032) — no disguised label proxies found.

In [6]:
import json
methodology_summary = {
    "features": ["march_impressions", "march_clicks", "march_position"],
    "label": "target_decline = 1 if april_impressions < march_impressions else 0",
    "model": "DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)",
    "split": "GroupShuffleSplit by client_hash_id, test_size=0.22, random_state=42",
    "leakage_test_week3": {"feature": "gsc_clicks", "honest_auc": 0.8756, "leaky_auc": 1.0},
    "leakage_test_week6": {"feature": "april_impressions", "honest_ap": 0.6114, "leaky_ap": 0.9909}
}
print(json.dumps(methodology_summary, indent=2))

{
  "features": [
    "march_impressions",
    "march_clicks",
    "march_position"
  ],
  "label": "target_decline = 1 if april_impressions < march_impressions else 0",
  "model": "DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)",
  "split": "GroupShuffleSplit by client_hash_id, test_size=0.22, random_state=42",
  "leakage_test_week3": {
    "feature": "gsc_clicks",
    "honest_auc": 0.8756,
    "leaky_auc": 1.0
  },
  "leakage_test_week6": {
    "feature": "april_impressions",
    "honest_ap": 0.6114,
    "leaky_ap": 0.9909
  }
}


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results

All numbers below come from the same held-out test set (11 clients never
seen in training), evaluated with Average Precision.

In [2]:
import pandas as pd

results_table = pd.DataFrame({
    "Method": [
        "Base rate (naive)",
        "Week-4 baseline (rule, recomputed)",
        "Decision Tree (naive random split)",
        "Decision Tree (honest grouped split)"
    ],
    "Average Precision": [0.5776, 0.5873, 0.6740, 0.6114]
})
print(results_table)

                                 Method  Average Precision
0                     Base rate (naive)             0.5776
1    Week-4 baseline (rule, recomputed)             0.5873
2    Decision Tree (naive random split)             0.6740
3  Decision Tree (honest grouped split)             0.6114


**Reading the table honestly:**
- The model measurably outperforms both the base rate (+0.034) and the
  recomputed Week-4 rule (+0.024) under the honest grouped split.
- The naive random split (0.674) looks stronger than the honest grouped
  split (0.611) — this gap (0.063) is itself a finding: it shows how much
  client-level memorization was inflating the naive number, and why the
  grouped estimate (0.611) is the one that should be trusted for unseen
  clients.
- Feature importance (grouped-split model): march_impressions (0.42),
  march_position (0.31), march_clicks (0.27) — all three contribute, none
  dominates completely.

## 5. Limitations

*What this work cannot claim.*
### Limitations

- **Observational, not experimental.** Nothing here proves that refreshing
  a page causes a change in performance — only that certain March signals
  are associated with an April decline.
- **Modest skill margin.** 0.611 vs a 0.578 base rate is a real but small
  improvement — this is decision-support, not a high-confidence oracle.
- **Coarse ranking granularity.** The depth-3 tree produces only a handful
  of distinct probability values, so "priority rank" should be read as
  priority *tiers*, not a strict 1-to-N order.
- **Narrow feature set.** Three March signals cannot capture content
  quality, competitor moves, or algorithm changes — a page can decline for
  reasons invisible to this model.
- **Limited population.** Trained on 36 clients, tested on 11, all from one
  17-month warehouse snapshot — not validated against clients or time
  periods outside this dataset.
- **Global time window.** A single March/April window was used for all
  clients, without checking each client's individual history depth
  (gsc_data_start), per the flyrank-data skill's panel warning.

In [7]:
print("Training clients:", 36)
print("Held-out test clients:", 11)
print("Total unique clients in this snapshot:", 47)
print("This model has not been evaluated on clients outside these 47.")

Training clients: 36
Held-out test clients: 11
Total unique clients in this snapshot: 47
This model has not been evaluated on clients outside these 47.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked recommendations (from the Week-7 action playbook)

The validated model scores each held-out page with a decline probability,
and pages are grouped into simple, human-readable priority tiers with
plain-language reason codes (not a black-box score alone).

**Top recommendation type:** "High-visibility, weak rank" pages — pages
that still carry meaningful visibility but show signs of slipping. This is
the closest match, in this project's own data, to the research paper's
Finding #4 pattern (31-90 day freshness window, 7.88:1 growth-to-decline
ratio) — offered as a directional rationale for prioritization, not a
causal guarantee.

**Human review is mandatory before any action.** No-go conditions include:
brand-new pages, very low impression volume (bottom 10%, ~4,438 of 43,247
pages flagged), and pages with a known external cause for their change
(campaign pause, migration, seasonality) — the model cannot see any of
these.

**Never automated:** publishing changes, deindexing, or using the score as
a performance metric for a person — every action requires human sign-off.

**Monitoring:** retrain if measured Average Precision drifts back toward
the base rate, if the client/content mix shifts materially, or on a
routine quarterly cadence.

In [8]:
low_volume_flagged = 4438
total_test_pages = 43247
print(f"Low-volume flagged pages: {low_volume_flagged} / {total_test_pages} "
      f"({low_volume_flagged/total_test_pages:.1%})")

Low-volume flagged pages: 4438 / 43247 (10.3%)


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [3]:
import matplotlib.pyplot as plt
import os

os.makedirs("work/figures", exist_ok=True)

# Chart 1: model vs baseline vs base rate
fig, ax = plt.subplots(figsize=(6, 4))
methods = ["Base rate", "Week-4 baseline", "Naive split\n(leakage risk)", "Honest\ngrouped split"]
values = [0.5776, 0.5873, 0.6740, 0.6114]
colors = ["#999999", "#6699cc", "#cc6666", "#339966"]
ax.bar(methods, values, color=colors)
ax.set_ylabel("Average Precision")
ax.set_title("Model vs. baseline vs. base rate (held-out clients)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("work/figures/results_comparison.png", dpi=150)
plt.close()
print("Saved: work/figures/results_comparison.png")

Saved: work/figures/results_comparison.png


This chart is the paper's primary Results figure. Caption to use on the
page: "The honest grouped-split model (0.611) beats both the base rate
(0.578) and the recomputed rule baseline (0.587); the naive random split
(0.674) shows how much a leaky evaluation would have overstated
performance."

## ML-12: Demo, social post, and employer summary

### 5-minute demo outline

If asked to walk someone through this project live, in this order:

**0:00–0:30 — The question**
"Content teams have thousands of pages and can't review them all. I built
a system to rank which pages are most likely to lose search visibility
next month, so an editor knows where to look first."

**0:30–1:30 — The data**
"176,738 real pages from FlyRank's production search warehouse — one
month of search signals (impressions, clicks, position) used to predict
whether the next month declines. Not a toy dataset."

**1:30–2:30 — The honest split (the core lesson)**
"Here's the trap: if I split train/test randomly, I get 0.674. But pages
from the same client leak information across the split — it's an
inflated number. Once I group the split by client, so no client is in
both train and test, the honest score is 0.611. That 0.06 gap IS the
finding — it's the cost of not doing this right."
(Action: open work/figures/results_comparison.png here and point at the
gap between the "naive split" and "honest grouped split" bars.)

**2:30–3:30 — Does it beat a simple rule?**
"A simple rule — just rank by log(impressions) — already scores 0.587.
The model beats it, but only by about 0.024. That's a real, measured
improvement, not a dramatic one — and I say that plainly rather than
oversell it."

**3:30–4:15 — What it's for, and what it's not**
"This is decision-support: a shortlist for a human editor, with reason
codes and a mandatory review list — never an auto-publish or auto-delete
tool. I show exactly which pages need extra caution (low volume, brand
new) before anyone acts on them."

**4:15–5:00 — Close**
"The whole pipeline — data contract, leakage hunts, honest validation,
and this action playbook — is public and reproducible."
(Action: open https://shahd799.github.io/flyrank-internship-1/ and scroll
to the Reproducibility section.)


### Social-post cut (LinkedIn-style)

---

**I built a model to rank which web pages will lose search visibility
next month — and the most useful number in the whole project was the one
that made my model look WORSE.**

Using 176,738 real pages from a production search warehouse, I trained a
small decision tree to flag pages likely to decline in search impressions
next month.

A naive train/test split scored 0.674 (Average Precision). Looked great.

Then I split honestly — by client, so no client's pages leaked between
train and test — and the real number was 0.611.

That 0.06 gap isn't a bug. It's the actual cost of getting validation
wrong, made visible instead of hidden.

The honest model still beats a transparent rule-based baseline (0.587)
and the base rate (0.578) — a real, modest, defensible improvement, wrapped
in a human-review playbook, not a black box.

Full paper + reproducible notebooks:
https://shahd799.github.io/flyrank-internship-1/

#MachineLearning #SEO #DataScience #MLOps

(Attach the results_comparison.png chart from work/figures/ when posting.)

---

### Employer-facing summary (3 sentences)

I built and validated a content-refresh prioritization model on 176,738
real pages from a production search analytics warehouse, using a
client-grouped train/test split to avoid inflated performance estimates —
a distinction I demonstrate explicitly by showing the naive split's score
(0.674) against the honest one (0.611). The final model measurably
outperforms both a transparent rule-based baseline (0.587) and the naive
base rate (0.578), and ships as a human-reviewed action playbook with
explicit reason codes, no-go conditions, and retrain triggers rather than
a bare prediction. The entire pipeline — data contract, two independent
leakage-injection tests, and the deployed research paper — is public and
reproducible from a single GitHub repository.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
